# 3. Score: ground truth from LiDAR, moving-object labels, error metrics (CPU, high RAM)

Needs the predictions of `02_predict_gpu`. For every block in `BLOCKS`: download camera and LiDAR layers once
(LiDAR archives are decompressed on all cores and every archive is checked against its published checksum), run
the dataset toolkit's validator, build the ground truth with the C++ core (Python if it is not built), label
moving and parked objects from the 3D boxes, score every scene for both models, save one pair of result files per
block, delete the block. Scenes are scored in parallel, one per CPU core; the numbers do not depend on that.

A finished block is skipped, so a dropped session simply continues. A block that fails the validator saves nothing.
Scenes with fewer than six Ouster LiDARs are skipped, so that ground-truth density is comparable.

Use a **CPU server with high RAM** (8 cores). Measured: 6.5 minutes for a block from a 6-LiDAR recording, about
9 minutes for one from a 12-LiDAR recording. To halve the wall-clock time, run `03_score_second_server` on a second
server at the same time.

**About the saved output below.** It is the record of the run that scored these blocks, and it is older than the last round of speed-ups (checksums of all archives at once, saved predictions fetched during the download, the validator in the background): its 12-LiDAR blocks took 12 to 17 minutes. The same code path, run later for the test split (`06_test_score`), took about 9 minutes for such a block, which is the figure quoted above. The scores do not depend on any of this.

**Next:** `04_report`.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
BLOCKS = [("train", 11), ("train", 12), ("train", 13),              # dark and wet: every scene is both (two recordings of one evening)
          ("train", 92), ("train", 82), ("train", 83), ("train", 90),   # motorway-rich
          ("train", 5), ("train", 6), ("train", 9),                 # wet in DAYLIGHT, to tell rain from darkness
          ("val", 0), ("val", 1), ("val", 2), ("val", 10), ("val", 12)]   # the validation blocks
# Chosen from the census for rare conditions, not at random. Finished blocks are skipped, so the list can grow.
# ("val", 11) is the development block (notebooks/development). The test blocks have their own notebooks, 05 to 07.
LIDAR_POLICY = "ouster_only"          # same six sensors in every scene, so ground-truth density is comparable
VALIDATE_DOWNLOAD = True              # run the dataset toolkit's own validator on each new block
REVERSE = False                       # forwards here; `03_score_second_server` walks the same list backwards,
                                      # so two CPU servers can share the work. This notebook alone does everything.
WORKERS = None                        # scenes scored at once. None = one per CPU core (max 8). The numbers do not depend on it

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

# build_cpp=True compiles the C++ geometry core on this server (about 15 s). Ground truth is then built
# with it, which gives exactly the same result as the Python reference, faster. If the build fails,
# everything still runs, in Python.
session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False, build_cpp=True)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
$ cmake -S /content/vggt-omega-aura-benchmark/cpp -B /content/vggt-omega-aura-benchmark/cpp/build -DCMAKE_BUILD_TYPE=Release -Dpybind11_DIR=/usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11 -DPython_EXECUTABLE=/usr/bin/python3
$ cmake --build /content/vggt-omega-aura-benchmark/cpp/build --config Release -j
C++ core    : built


In [4]:
# --- 4. Process the blocks ---
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

pd.set_option("display.width", 220)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")   # faulty scenes the maintainers exclude
print("scenes excluded by the dataset:", len(EXCLUDED))
available = ad.available_blocks(chunks, scene_blocks, hub_files, [pl.CAMERA_LAYER, pl.LIDAR_LAYER])
import uuid
ME = ("backward-" if REVERSE else "forward-") + uuid.uuid4().hex[:6]      # this server's name on its claims
summaries, left_to_the_other = [], []
for split, block in (list(reversed(BLOCKS)) if REVERSE else list(BLOCKS)):
    if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, split, block) \
            and not pl.claim_block(session.persist_root, RUN_TAG, split, block, ME, max_age_s=1500):
        print(f"=== {pl.block_tag(split, block)}: the other server is on it, skipped ===")
        left_to_the_other.append((split, block))
        continue
    assert (split, block) in available.index, f"block {(split, block)} is not downloadable with camera + LiDAR"
    print(f"=== {pl.block_tag(split, block)} ({available.loc[(split, block), 'total_gb']} GB) ===")
    try:
        summary = pl.process_block(session, split, block, ad.block_scene_ids(scene_blocks, split, block, EXCLUDED), CAMERA, MODELS,
                                   RUN_TAG, lidar_policy=LIDAR_POLICY, validate=VALIDATE_DOWNLOAD,
                                   scene_names=ad.block_scene_names(scene_blocks, split, block, EXCLUDED), workers=WORKERS)
    finally:                 # a claim must not outlive a crash: a re-run gets a new name and would wait for it
        pl.release_claim(session.persist_root, RUN_TAG, split, block, ME)
    print(" ", summary)
    summaries.append(summary)
print()
print(pd.DataFrame([{k: v for k, v in s.items() if k not in ("sensors", "validation")} for s in summaries]).to_string(index=False))
still_open = [b for b in left_to_the_other if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, *b)]
if still_open:
    print()
    print("left to the other server and not finished yet:", still_open, "| if that server stopped, run this notebook again")

scenes excluded by the dataset: 8
=== train_block000011 (11.77 GB) ===
  {'block': 'train_block000011', 'status': 'already done'}
=== train_block000012 (12.29 GB) ===
  {'block': 'train_block000012', 'status': 'already done'}
=== train_block000013 (12.37 GB) ===
  {'block': 'train_block000013', 'status': 'already done'}
=== train_block000092 (8.41 GB) ===
  {'block': 'train_block000092', 'status': 'already done'}
=== train_block000082 (8.01 GB) ===
  {'block': 'train_block000082', 'status': 'already done'}
=== train_block000083 (8.3 GB) ===
  {'block': 'train_block000083', 'status': 'already done'}
=== train_block000090 (8.46 GB) ===
  {'block': 'train_block000090', 'status': 'already done'}
=== train_block000005 (16.01 GB) ===
  downloading with the toolkit, decompressing with xz on all 8 cores


  fast unpack: {'archives': 5, 'xz_decompressed_on_all_cores': 3, 'download_s': 43.9, 'verify_and_decompress_s': 251.1, 'extract_s': 123.3}
  validator: {'ok': True, 'scenes_checked': 20, 'errors': []}
  predictions: 0 made now, the rest loaded (77 s) | scoring 20 scenes with 8 worker(s)
  2026-06-02-17-05-20|138     56.8 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|1       80.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-10-44-05|25      60.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|141     57.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-09-57-03|59      66.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-15-28-52|37      84.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|19      78.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cp

  fast unpack: {'archives': 4, 'xz_decompressed_on_all_cores': 2, 'download_s': 41.3, 'verify_and_decompress_s': 240.6, 'extract_s': 147.2}
  validator: {'ok': True, 'scenes_checked': 20, 'errors': []}
  predictions: 0 made now, the rest loaded (78 s) | scoring 20 scenes with 8 worker(s)
  2026-06-02-16-19-19|11      56.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-19-19|32      55.8 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-14-28-29|49      61.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-14-28-29|44      41.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-19-19|36      54.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-15-30-32|63      55.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-15-30-32|68      62.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cp

  fast unpack: {'archives': 4, 'xz_decompressed_on_all_cores': 2, 'download_s': 75.2, 'verify_and_decompress_s': 277.2, 'extract_s': 363.5}
  validator: {'ok': True, 'scenes_checked': 20, 'errors': []}
  predictions: 0 made now, the rest loaded (65 s) | scoring 20 scenes with 8 worker(s)
  2026-06-02-14-28-29|48      49.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-05-28-16-26-51|145     65.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|80      56.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|38      54.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|3       61.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-16-36-33|6       44.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-11-25-54|27      40.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cp

In [5]:
# --- 5. What the run holds so far ---
for model in MODELS:
    rows, scenes = pl.load_run(session.persist_root, RUN_TAG, model)
    print(f"{model}: {scenes['scene_id'].nunique() if len(scenes) else 0} scenes in "
          f"{scenes[['split', 'block']].drop_duplicates().shape[0] if len(scenes) else 0} blocks")

vggt_omega_512: 287 scenes in 15 blocks
vggt_1b: 287 scenes in 15 blocks
